# HVTB HPA Paper Metrics

Notebook này dành cho HVTB để chấm nhánh TrOCR HPA trên GT crops/submission CSV và xuất số liệu điền paper. Metric official được copy nguyên từ notebook BTC `official-evaluation-metric-text-normalization.ipynb`; các metric HPA crop-level thêm vào gồm CER, WER, Exact Match theo overall và từng type `handwritten/printed/annotation`.

Output chính nằm trong `fillpaper/evaluate/hvtb_hpa_metric_outputs/`.


In [ ]:
from pathlib import Path


def get_evaluator_root():
    """Folder đặt notebook chấm điểm.

    Trong Jupyter/VS Code thông thường, Path.cwd() chính là folder chứa notebook.
    Nếu môi trường có expose đường dẫn notebook qua __vsc_ipynb_file__ thì ưu tiên dùng nó.
    Nếu kernel của bạn đang chạy ở cwd khác, chỉ cần sửa EVALUATOR_ROOT thành Path(r"...").resolve().
    """
    notebook_file = globals().get("__vsc_ipynb_file__")
    if notebook_file:
        return Path(notebook_file).resolve().parent
    return Path.cwd().resolve()


EVALUATOR_ROOT = get_evaluator_root()
GT_JSONL = EVALUATOR_ROOT / "test.jsonl"
OUTPUT_DIR = EVALUATOR_ROOT / "hvtb_hpa_metric_outputs"

# Ba folder CSV đặt cùng folder với notebook chấm điểm này.
# Notebook đọc mọi file .csv trong các folder này, không phụ thuộc tên file.
TABLE_RESULT_DIRS = {
    "module_results": EVALUATOR_ROOT / "module_results",
    "hpa_ablation": EVALUATOR_ROOT / "hpa_ablation",
    "per_type_results": EVALUATOR_ROOT / "per_type_results",
}
SUBMISSION_SEARCH_DIRS = list(TABLE_RESULT_DIRS.values())

EXTRA_SUBMISSION_FILES = [
    # Path(r"C:/path/to/02_hpa_gt_crops_submission.csv"),
]

# Run này dùng để tạo 6 dòng TrOCR HPA trong tab:module-results.
FINAL_HPA_RUN_ID = "02_hpa_gt_crops"

ROW_ID_COLUMN = "image"
IOU_THRESHOLD = 0.5
ROUND_DIGITS = 4

HPA_TYPES = ["handwritten", "printed", "annotation"]
HPA_ABLATION_ORDER = [
    "zero-shot",
    "gold-only",
    "silver warm-up",
    "silver-to-gold",
    "full curriculum",
    "uniform decoding",
    "type-specific decoding",
    "TrOCR HPA final GT crops",
]

RUN_LABEL_OVERRIDES = {
    "02_hpa_gt_crops": "TrOCR HPA final GT crops",
}

print("EVALUATOR_ROOT:", EVALUATOR_ROOT)
print("GT_JSONL:", GT_JSONL)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CSV folders:")
for name, path in TABLE_RESULT_DIRS.items():
    print(f"- {name}: {path}")


In [ ]:
import sys
from pathlib import Path

KAGGLE_METRIC_SOURCE = '"""\nCustom Kaggle evaluation metric for Ukrainian Handwritten Text Recognition.\n\nScore = 0.15 * Detection_F1 + 0.05 * ClassAcc + 0.30 * (1 - CER) + 0.50 * (1 - PageCER)\n\nText is normalized before CER comparison:\n  - Cyrillic/Latin lookalike characters → Cyrillic\n  - All dash types → hyphen-minus\n  - Whitespace collapse, strip\n  - Quote/apostrophe normalization\n  - Strikethrough markers ~~text~~ removed\n  - Formula: Unicode super/subscripts → ^/_ notation, single-char braces removed\n  - Tables: whitespace around pipes stripped\n\nSubmission format: CSV with columns `image` and `regions`.\nThe `regions` column contains a JSON-encoded list of region objects:\n    [{"bbox": [x1, y1, x2, y2], "type": "handwritten", "text": "..."}]\n\nRequired fields per region: bbox, type, text.\nRegion types: handwritten, printed, formula, table, annotation, image, graph.\nIf an image has no regions, use an empty list: []\n\nNote: `language` and `legibility` are GT-only attributes — participants do not need to predict them.\n\nRecent changes\n--------------\n- PageCER is now computed symmetrically: prediction regions that match a\n  non-scorable GT region (illegible / language=other / image / graph)\n  are excluded from `pred_page`, in the same way GT non-scorable regions\n  are excluded from `gt_page`. Previously, a prediction\'s text for an\n  illegible GT region inflated `pred_page` while the GT side excluded it,\n  preventing theoretically-perfect scores on pages with such regions.\n  Detection F1, classification accuracy and per-region CER are unchanged.\n\n>>> import pandas as pd\n>>> row_id_column_name = "image"\n>>> solution = pd.DataFrame({\n...     "image": ["test.jpg"],\n...     "regions": [\'[{"bbox":[50,100,850,130],"type":"handwritten","language":"uk","legibility":"legible","text":"Доброго ранку"},{"bbox":[50,150,870,180],"type":"handwritten","language":"uk","legibility":"legible","text":"Сьогодні гарна погода"},{"bbox":[50,250,650,270],"type":"printed","language":"uk","legibility":"legible","text":"Завдання 1"},{"bbox":[700,50,950,250],"type":"image","language":"uk","legibility":"legible","text":""},{"bbox":[900,950,980,990],"type":"annotation","language":"uk","legibility":"legible","text":"5"}]\'],\n... })\n>>> submission = pd.DataFrame({\n...     "image": ["test.jpg"],\n...     "regions": [\'[{"bbox":[52,101,852,131],"type":"handwritten","language":"uk","legibility":"legible","text":"Доброго ранку"},{"bbox":[50,148,860,184],"type":"handwritten","language":"uk","legibility":"legible","text":"Сьогодні гарна пагода"},{"bbox":[48,249,649,272],"type":"handwritten","language":"uk","legibility":"legible","text":"Завдання 1"},{"bbox":[710,60,945,248],"type":"image","language":"uk","legibility":"legible","text":""}]\'],\n... })\n>>> round(score(solution, submission, row_id_column_name), 4)\n0.9348\n"""\n\nimport json\nimport re\nimport pandas as pd\n\n\nclass ParticipantVisibleError(Exception):\n    pass\n\n\n# ── Text normalization ────────────────────────────────────────\n\n# ── LaTeX command → Unicode symbol mapping ────────────────────\n# Applied BEFORE Cyrillic/Latin conversion (so \\pi doesn\'t become \\рі)\n\n_LATEX_SYMBOLS = {\n    # Greek letters (common in formulas)\n    r\'\\alpha\': \'α\', r\'\\beta\': \'β\', r\'\\gamma\': \'γ\', r\'\\delta\': \'δ\',\n    r\'\\epsilon\': \'ε\', r\'\\varepsilon\': \'ε\', r\'\\zeta\': \'ζ\', r\'\\eta\': \'η\',\n    r\'\\theta\': \'θ\', r\'\\vartheta\': \'ϑ\', r\'\\iota\': \'ι\', r\'\\kappa\': \'κ\',\n    r\'\\lambda\': \'λ\', r\'\\mu\': \'μ\', r\'\\nu\': \'ν\', r\'\\xi\': \'ξ\',\n    r\'\\pi\': \'π\', r\'\\rho\': \'ρ\', r\'\\sigma\': \'σ\', r\'\\tau\': \'τ\',\n    r\'\\upsilon\': \'υ\', r\'\\phi\': \'φ\', r\'\\varphi\': \'φ\', r\'\\chi\': \'χ\',\n    r\'\\psi\': \'ψ\', r\'\\omega\': \'ω\',\n    r\'\\Gamma\': \'Γ\', r\'\\Delta\': \'Δ\', r\'\\Theta\': \'Θ\', r\'\\Lambda\': \'Λ\',\n    r\'\\Xi\': \'Ξ\', r\'\\Pi\': \'Π\', r\'\\Sigma\': \'Σ\', r\'\\Phi\': \'Φ\',\n    r\'\\Psi\': \'Ψ\', r\'\\Omega\': \'Ω\',\n    # Operators & relations\n    r\'\\cdot\': \'·\', r\'\\times\': \'×\', r\'\\div\': \'÷\', r\'\\pm\': \'±\', r\'\\mp\': \'∓\',\n    r\'\\circ\': \'∘\', r\'\\bullet\': \'•\', r\'\\star\': \'⋆\',\n    r\'\\leq\': \'≤\', r\'\\le\': \'≤\', r\'\\geq\': \'≥\', r\'\\ge\': \'≥\',\n    r\'\\neq\': \'≠\', r\'\\ne\': \'≠\', r\'\\approx\': \'≈\', r\'\\equiv\': \'≡\',\n    r\'\\sim\': \'∼\', r\'\\propto\': \'∝\',\n    # Arrows\n    r\'\\rightarrow\': \'→\', r\'\\to\': \'→\', r\'\\leftarrow\': \'←\',\n    r\'\\leftrightarrow\': \'↔\', r\'\\Rightarrow\': \'⇒\', r\'\\Leftarrow\': \'⇐\',\n    r\'\\Leftrightarrow\': \'⇔\', r\'\\implies\': \'⇒\', r\'\\iff\': \'⇔\',\n    # Set theory\n    r\'\\cap\': \'∩\', r\'\\cup\': \'∪\', r\'\\subset\': \'⊂\', r\'\\supset\': \'⊃\',\n    r\'\\subseteq\': \'⊆\', r\'\\supseteq\': \'⊇\', r\'\\in\': \'∈\', r\'\\notin\': \'∉\',\n    r\'\\emptyset\': \'∅\', r\'\\varnothing\': \'∅\',\n    r\'\\oplus\': \'⊕\', r\'\\otimes\': \'⊗\',\n    # Misc\n    r\'\\infty\': \'∞\', r\'\\partial\': \'∂\', r\'\\nabla\': \'∇\',\n    r\'\\forall\': \'∀\', r\'\\exists\': \'∃\', r\'\\neg\': \'¬\',\n    r\'\\sqrt\': \'√\', r\'\\sum\': \'∑\', r\'\\prod\': \'∏\', r\'\\int\': \'∫\',\n    r\'\\ldots\': \'…\', r\'\\dots\': \'…\', r\'\\cdots\': \'⋯\',\n    # Geometry / logic symbols\n    r\'\\therefore\': \'∴\', r\'\\because\': \'∵\',\n    r\'\\perp\': \'⊥\', r\'\\angle\': \'∠\', r\'\\parallel\': \'∥\',\n    r\'\\square\': \'□\', r\'\\Box\': \'□\', r\'\\triangle\': \'△\',\n    # Up/down arrows (often used as products of reaction)\n    r\'\\uparrow\': \'↑\', r\'\\downarrow\': \'↓\', r\'\\Uparrow\': \'⇑\', r\'\\Downarrow\': \'⇓\',\n    # Logical operators\n    r\'\\vee\': \'∨\', r\'\\wedge\': \'∧\', r\'\\lor\': \'∨\', r\'\\land\': \'∧\',\n    # Set operations\n    r\'\\setminus\': \'∖\', r\'\\backslash\': \'\\\\\',\n    # Vertical bar variants\n    r\'\\mid\': \'|\', r\'\\lvert\': \'|\', r\'\\rvert\': \'|\', r\'\\Vert\': \'‖\',\n    # Floor/ceil delimiters\n    r\'\\lfloor\': \'⌊\', r\'\\rfloor\': \'⌋\', r\'\\lceil\': \'⌈\', r\'\\rceil\': \'⌉\',\n    # Marvosym/wasysym symbols (used in genetics for sex notation)\n    r\'\\male\': \'♂\', r\'\\female\': \'♀\',\n}\n\n# LaTeX named functions: \\sin, \\cos, \\ln, \\lim, ... → strip backslash + trailing space\n# Without trailing space "\\\\sin\\\\alpha" would normalize to "sinα" while "sin α" → "sin α"\n# (mismatched). Trailing space collapses with following whitespace via _MULTI_SPACE.\n_LATEX_FUNCTION_NAMES = (\n    \'arcsin\',\'arccos\',\'arctan\',\'arcctg\',\'arccot\',\'arcsec\',\'arccsc\',\n    \'sinh\',\'cosh\',\'tanh\',\'coth\',\n    \'sin\',\'cos\',\'tan\',\'cot\',\'sec\',\'csc\',\'ctg\',\n    \'liminf\',\'limsup\',\n    \'log\',\'ln\',\'lg\',\'exp\',\'lim\',\'sup\',\'inf\',\'min\',\'max\',\'det\',\'dim\',\'gcd\',\'lcm\',\'mod\',\n    \'arg\',\'deg\',\'hom\',\'ker\',\n)\n# `(?![A-Za-z])` (not `\\b`) so that `\\lim_{x→0}` and `\\sin{x}` still match: the\n# char after the command name may be `_`/`^`/`{` which `\\b` would block.\n_LATEX_FUNCTIONS_RE = re.compile(r\'\\\\(\' + \'|\'.join(_LATEX_FUNCTION_NAMES) + r\')(?![A-Za-z])\')\n\n# \\xrightarrow{label} / \\xleftarrow{label} → arrow (label discarded)\n# Some forms have optional bracketed below-label: \\xrightarrow[below]{above}\n_XARROW_RIGHT = re.compile(r\'\\\\xrightarrow\\s*(?:\\[[^\\]]*\\])?\\s*\\{[^{}]*\\}\')\n_XARROW_LEFT = re.compile(r\'\\\\xleftarrow\\s*(?:\\[[^\\]]*\\])?\\s*\\{[^{}]*\\}\')\n\n# Sizing commands (visual hints, no semantic value) → strip\n_LATEX_SIZING = re.compile(r\'\\\\(?:big|Big|bigg|Bigg)[lr]?\\b\')\n\n# Math styles \\mathrm{}, \\mathbf{}, ..., \\operatorname{} → strip wrapper\n_MATH_STYLE = re.compile(r\'\\\\(?:mathrm|mathbf|mathit|mathbb|mathcal|mathfrak|mathsf|mathtt|operatorname|boldsymbol|pmb)\\s*\\{([^{}]*)\\}\')\n\n# Cancel/overset/underset: cancellation marks → keep base content\n# \\cancel{18} → 18 (the struck-through value is what we want to read back)\n# \\overset{a}{b} → b (b is the main symbol; a is a decoration like an oxidation state)\n# \\underset{a}{b} → b (same logic)\n_CANCEL = re.compile(r\'\\\\cancel\\s*\\{([^{}]*)\\}\')\n_OVERSET = re.compile(r\'\\\\overset\\s*\\{[^{}]*\\}\\s*\\{([^{}]*)\\}\')\n_UNDERSET = re.compile(r\'\\\\underset\\s*\\{[^{}]*\\}\\s*\\{([^{}]*)\\}\')\n# Sort by length descending so \\rightarrow matches before \\right\n_LATEX_COMMANDS_RE = re.compile(\n    \'|\'.join(re.escape(k) for k in sorted(_LATEX_SYMBOLS.keys(), key=len, reverse=True))\n)\n\n# \\text{...} → content (strip LaTeX text wrapper)\n_LATEX_TEXT_WRAPPER = re.compile(r\'\\\\text\\{([^}]*)\\}\')\n# \\left( \\right) → ( )\n_LATEX_LEFT_RIGHT = re.compile(r\'\\\\(left|right)\\s*([()|\\[\\]{}.])\')\n# LaTeX spacing commands → space or nothing\n_LATEX_SPACING = re.compile(r\'\\\\[,;:!]|\\\\quad|\\\\qquad|\\\\hspace\\{[^}]*\\}\')\n\n# Multiplication sign normalization: * and · → · (middle dot)\n_MULT_SIGNS = re.compile(r\'[*∗⋅]\')  # asterisk, combining asterisk, dot operator\n\n# Latin → Cyrillic lookalike mapping (lowercase + uppercase)\n_LATIN_TO_CYRILLIC = {\n    \'a\': \'а\', \'c\': \'с\', \'e\': \'е\', \'i\': \'і\', \'o\': \'о\',\n    \'p\': \'р\', \'x\': \'х\', \'y\': \'у\',\n    \'A\': \'А\', \'B\': \'В\', \'C\': \'С\', \'E\': \'Е\', \'H\': \'Н\',\n    \'K\': \'К\', \'M\': \'М\', \'O\': \'О\', \'P\': \'Р\', \'T\': \'Т\', \'X\': \'Х\',\n}\n\n# Unicode superscript/subscript → ASCII\n_SUPERSCRIPTS = str.maketrans(\'⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻⁼⁽⁾ⁿ\', \'0123456789+-=()n\')\n_SUBSCRIPTS = str.maketrans(\'₀₁₂₃₄₅₆₇₈₉₊₋₌₍₎\', \'0123456789+-=()\')\n\n# All dash-like characters → hyphen-minus\n_DASHES = re.compile(r\'[\\u2010\\u2011\\u2012\\u2013\\u2014\\u2015\\u2212\\uFE58\\uFE63\\uFF0D]\')\n\n# Filler dashes/underscores (3+ repeated) → normalized form\n_FILLERS = re.compile(r\'[_\\-]{3,}\')\n\n# Strikethrough: ~~old~~{new} → new (correction replaces struck-through text)\n_STRIKETHROUGH_CORRECTION = re.compile(r\'~~.*?~~\\{(.*?)\\}\')\n# Strikethrough: ~~text~~ → text (standalone, no correction)\n_STRIKETHROUGH = re.compile(r\'~~(.*?)~~\')\n\n# Multiple whitespace → single space\n_MULTI_SPACE = re.compile(r\'[ \\t\\u00A0\\u2000-\\u200B\\u3000]+\')\n\n# Spaces inside brackets: "( text )" → "(text)"\n_SPACE_IN_PARENS = re.compile(r\'\\(\\s+\')\n_SPACE_IN_PARENS_R = re.compile(r\'\\s+\\)\')\n\n# Space immediately before "(" or before sub/superscript markers — strips\n# the trailing space introduced by `\\sin ` / `\\lim ` etc. so:\n#   "\\sin(x)" → "sin (x)" → "sin(x)" (matches plain "sin(x)")\n#   "\\lim_{x→0}" → "lim _{x→0}" → "lim_{x→0}"\n_SPACE_BEFORE_PAREN = re.compile(r\' +\\(\')\n_SPACE_BEFORE_SUBSUPER = re.compile(r\' +([_^])\')\n\n# LaTeX braces: x_{3} → x_3, x^{2} → x^2, S_{повн} → S_повн\n_LATEX_BRACE = re.compile(r\'([_^])\\{([^}]+)\\}\')\n\n# LaTeX table environments → PSV\n# Covers: array, tabular, matrix/pmatrix/bmatrix/vmatrix/Vmatrix/smallmatrix,\n# aligned/align/alignat/gathered/cases (multi-row alignment envs that share\n# the same \\\\ row-separator + & column-separator syntax).\n_LATEX_TABLE_ENV = re.compile(r\'\\\\begin\\{(?:array|tabular|matrix|pmatrix|bmatrix|vmatrix|Vmatrix|smallmatrix|aligned|align|alignat|gathered|cases|split)\\*?\\}(?:\\{[^}]*\\})?\\s*\')\n_LATEX_TABLE_ENV_END = re.compile(r\'\\s*\\\\end\\{(?:array|tabular|matrix|pmatrix|bmatrix|vmatrix|Vmatrix|smallmatrix|aligned|align|alignat|gathered|cases|split)\\*?\\}\')\n_LATEX_TABLE_ROW_SEP = re.compile(r\'\\s*\\\\\\\\\\s*\')\n_LATEX_TABLE_COL_SEP = re.compile(r\'\\s*&\\s*\')\n\n# Table horizontal line decorations are visual-only, no semantic value.\n# `\\hline` is a no-arg command; `\\cline{2-4}` takes a span argument.\n_TABLE_LINES = re.compile(r\'\\\\hline\\b|\\\\cline\\s*\\{[^{}]*\\}\')\n\n# `\\phantom{x}` renders invisible — no semantic value, strip.\n_PHANTOM = re.compile(r\'\\\\phantom\\s*\\{[^{}]*\\}\')\n\n# `\\underline{x}` → x  (visual underline; same treatment as `\\bar` etc.)\n_UNDERLINE = re.compile(r\'\\\\underline\\s*\\{([^{}]*)\\}\')\n\n# Student-style row separator inside plain parens: `(a b \\n c d)` means a 2D\n# matrix (one row per `\\n`). Normalize to multiline PSV like the existing\n# `(a; b)` rule does for column matrices.\n# Match an opening paren, body containing at least one literal `\\n`, closing paren.\n_PAREN_NEWLINE_ROWS = re.compile(r\'\\(([^()]*\\\\n[^()]*)\\)\')\n\n# Quotes normalization\n_QUOTES_DOUBLE = re.compile(r\'["\\u201C\\u201D\\u201E\\u00AB\\u00BB\\u2033]\')\n_QUOTES_SINGLE = re.compile(r"[\'\\u2018\\u2019\\u02BC\\u0027\\u2032]")\n\n# Pipe-separated values: strip whitespace around pipes\n_PSV_PIPE = re.compile(r\'\\s*\\|\\s*\')\n\n# \\frac{a}{b} → a/b  (applied iteratively for nested fractions)\n_FRAC = re.compile(r\'\\\\frac\\s*\\{([^{}]*)\\}\\s*\\{([^{}]*)\\}\')\n\n# \\sqrt{...} → √...  (after \\sqrt → √ symbol mapping, strip the trailing argument braces)\n_SQRT_BRACES = re.compile(r\'√\\s*\\{([^{}]*)\\}\')\n\n# Arrow decorations: \\overrightarrow{X} → →X, \\overleftarrow{X} → ←X, \\vec{X} → →X\n_ARROW_RIGHT_DECOR = re.compile(r\'\\\\(?:overrightarrow|vec)\\s*\\{([^{}]*)\\}\')\n_ARROW_LEFT_DECOR = re.compile(r\'\\\\overleftarrow\\s*\\{([^{}]*)\\}\')\n\n# Decorators: \\bar{x}, \\hat{x}, \\overline{x}, \\widetilde{x}, \\widehat{x}, \\dot{x}, \\ddot{x} → strip wrapper\n_DECORATOR = re.compile(r\'\\\\(?:bar|hat|overline|widetilde|widehat|dot|ddot)\\s*\\{([^{}]*)\\}\')\n\n# Combining diacritical marks (U+0300-036F) + symbol combining (U+20D0-20FF, includes \\vec arrow ⃗)\n# Strip after decorators to symmetrize: \\bar{x} (stripped to x) ↔ x̄ (Unicode combining macron stripped to x)\n_COMBINING_MARKS = re.compile(r\'[̀-ͯ⃐-\u20ff]\')\n\n# Plain (a; b; c) column matrix → PSV (a\\nb\\nc)  — only when paren content is purely semicolon-separated\n_PAREN_SEMI = re.compile(r\'\\(([^();]+(?:\\s*;\\s*[^();]+)+)\\)\')\n\n# Plain pipe determinant: | a b | | c d | | e f | (2+ pipe-bounded segments) → PSV\n_PIPE_MATRIX = re.compile(r\'\\|\\s*([^|\\n]+?)\\s*\\|(?:\\s*\\|\\s*[^|\\n]+?\\s*\\|)+\')\n\n\ndef _normalize_text(text: str, region_type: str = "handwritten") -> str:\n    """\n    Normalize text before CER comparison.\n    Applied identically to both GT and prediction.\n\n    >>> _normalize_text("Доброго ранку")\n    \'Доброго ранку\'\n    >>> _normalize_text("cocна")  # Latin \'c\',\'o\' → Cyrillic\n    \'сосна\'\n    >>> _normalize_text("тире — довге")  # em-dash → hyphen\n    \'тире - довге\'\n    >>> _normalize_text("~~закреслено~~ слово")\n    \'закреслено слово\'\n    >>> _normalize_text("x_{3} + y^{2}", region_type="formula")\n    \'х_3 + у^2\'\n    >>> _normalize_text("A | B | C", region_type="table")\n    \'А|В|С\'\n    >>> _normalize_text("x² + y₃", region_type="formula")\n    \'х^2 + у_3\'\n    >>> _normalize_text(\'«Привіт»\')\n    \'"Привіт"\'\n    >>> _normalize_text("( дужки )")\n    \'(дужки)\'\n    >>> _normalize_text("_____")\n    \'___\'\n    >>> _normalize_text("cat") == _normalize_text("сat")  # Latin/Cyrillic forgiven\n    True\n    >>> _normalize_text("π r^2", region_type="formula")  # Unicode π unchanged\n    \'π r^2\'\n    >>> _normalize_text("2 * 3 = 6", region_type="formula")  # * → ·\n    \'2 · 3 = 6\'\n    >>> _normalize_text("x² + y₃", region_type="formula")  # Unicode super/sub\n    \'х^2 + у_3\'\n    >>> _normalize_text("H_{2}SO_{4}", region_type="formula")  # single-char braces\n    \'Н_2SО_4\'\n    >>> _normalize_text(r"\\\\frac{1}{2}", region_type="formula")  # frac → plain\n    \'1/2\'\n    >>> _normalize_text(r"a/b", region_type="formula") == _normalize_text(r"\\\\frac{a}{b}", region_type="formula")\n    True\n    >>> _normalize_text(r"\\\\sqrt{169}", region_type="formula")\n    \'√169\'\n    >>> _normalize_text(r"\\\\bar{x}", region_type="formula")\n    \'х\'\n    >>> _normalize_text("(1; -1)", region_type="formula")  # column matrix → PSV\n    \'1\\\\n-1\'\n    >>> _normalize_text("| 3 2 | | -1 1 |", region_type="formula")  # pipe determinant → PSV\n    \'3 2\\\\n-1 1\'\n    >>> _normalize_text(r"\\\\overrightarrow{AB}", region_type="formula")  # arrow notation\n    \'→АВ\'\n    >>> _normalize_text(r"\\\\vec{a}", region_type="formula")  # vec also → arrow\n    \'→а\'\n    >>> _normalize_text(r"\\\\therefore \\\\overrightarrow{AB} \\\\perp \\\\overrightarrow{AC}", region_type="formula")\n    \'∴ →АВ ⊥ →АС\'\n    >>> _normalize_text("x̄", region_type="formula") == _normalize_text(r"\\\\bar{x}", region_type="formula")\n    True\n    """\n    if not text:\n        return ""\n\n    # 1. Strikethrough: ~~old~~{new} → new (must come before plain ~~)\n    text = _STRIKETHROUGH_CORRECTION.sub(r\'\\1\', text)\n    text = _STRIKETHROUGH.sub(r\'\\1\', text)\n\n    # 2. LaTeX normalization (BEFORE Cyrillic conversion — so \\pi doesn\'t become \\рі)\n    if region_type in ("formula", "table"):\n        # LaTeX table environments → PSV (must be before symbol conversion)\n        text = _LATEX_TABLE_ENV.sub(\'\', text)\n        text = _LATEX_TABLE_ENV_END.sub(\'\', text)\n        text = _LATEX_TABLE_ROW_SEP.sub(\'\\n\', text)\n        text = _LATEX_TABLE_COL_SEP.sub(\'|\', text)\n        # Visual-only table decorations: \\hline, \\cline{2-4}, \\phantom{x}\n        text = _TABLE_LINES.sub(\'\', text)\n        text = _PHANTOM.sub(\'\', text)\n        # \\underline{x} → x (treat like \\bar)\n        text = _UNDERLINE.sub(r\'\\1\', text)\n        # \\text{...} → content\n        text = _LATEX_TEXT_WRAPPER.sub(r\'\\1\', text)\n        # \\left( \\right) → ( )\n        text = _LATEX_LEFT_RIGHT.sub(r\'\\2\', text)\n        # Sizing hints \\big, \\Bigg, \\bigl, \\biggr, … → strip\n        text = _LATEX_SIZING.sub(\'\', text)\n        # Math styles \\mathrm{}, \\mathbf{}, ..., \\operatorname{} → strip wrapper (iterate for nested)\n        prev = None\n        while text != prev:\n            prev = text\n            text = _MATH_STYLE.sub(r\'\\1\', text)\n        # \\xrightarrow[below]{above} / \\xleftarrow → → / ←\n        text = _XARROW_RIGHT.sub(\'→\', text)\n        text = _XARROW_LEFT.sub(\'←\', text)\n        # \\cancel{x} → x; \\overset{a}{b} / \\underset{a}{b} → b\n        # Iterate because these can be nested (e.g. \\overset{\\overset{...}{|}}{X})\n        prev = None\n        while text != prev:\n            prev = text\n            text = _CANCEL.sub(r\'\\1\', text)\n            text = _OVERSET.sub(r\'\\1\', text)\n            text = _UNDERSET.sub(r\'\\1\', text)\n        # LaTeX spacing → single space\n        text = _LATEX_SPACING.sub(\' \', text)\n        # \\frac{a}{b} → a/b  (iterate for nested fractions)\n        prev = None\n        while text != prev:\n            prev = text\n            text = _FRAC.sub(r\'\\1/\\2\', text)\n        # Arrow decorations: \\overrightarrow{X}/\\vec{X} → →X, \\overleftarrow{X} → ←X\n        text = _ARROW_RIGHT_DECOR.sub(r\'→\\1\', text)\n        text = _ARROW_LEFT_DECOR.sub(r\'←\\1\', text)\n        # Decorators: \\bar{x}, \\hat{x}, \\overline{x} → strip wrapper\n        text = _DECORATOR.sub(r\'\\1\', text)\n        # Named functions: \\sin, \\cos, \\log, \\lim … → "sin ", "cos ", … (trailing space\n        # ensures "\\\\sin\\\\alpha" → "sin α" matches "sin α"; collapsed later)\n        text = _LATEX_FUNCTIONS_RE.sub(r\'\\1 \', text)\n        # LaTeX symbols → Unicode (\\pi → π, \\cdot → ·, etc.)\n        text = _LATEX_COMMANDS_RE.sub(lambda m: _LATEX_SYMBOLS[m.group()], text)\n        # \\sqrt argument braces: √{169} → √169  (after \\sqrt → √ mapping above)\n        text = _SQRT_BRACES.sub(r\'√\\1\', text)\n        # Strip Unicode combining marks (symmetric with decorator wrapper-stripping above)\n        # e.g. x̄ (x + U+0304 macron) → x; a⃗ (a + U+20D7 right arrow) → a\n        text = _COMBINING_MARKS.sub(\'\', text)\n        # Multiplication signs: * ∗ ⋅ → · (middle dot)\n        text = _MULT_SIGNS.sub(\'·\', text)\n        # Unicode superscripts → ^N\n        converted = []\n        for ch in text:\n            if ch in \'⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻⁼⁽⁾ⁿ\':\n                converted.append(\'^\' + ch.translate(_SUPERSCRIPTS))\n            elif ch in \'₀₁₂₃₄₅₆₇₈₉₊₋₌₍₎\':\n                converted.append(\'_\' + ch.translate(_SUBSCRIPTS))\n            else:\n                converted.append(ch)\n        text = \'\'.join(converted)\n        # Braces: x_{3} → x_3, S_{повн} → S_повн\n        text = _LATEX_BRACE.sub(r\'\\1\\2\', text)\n        # Plain (a; b; c) column matrix → PSV\n        text = _PAREN_SEMI.sub(lambda m: \'\\n\'.join(x.strip() for x in m.group(1).split(\';\')), text)\n        # Plain `(a b \\n c d)` 2D matrix (student notation) → multiline rows\n        text = _PAREN_NEWLINE_ROWS.sub(lambda m: \'\\n\'.join(x.strip() for x in m.group(1).split(\'\\\\n\')), text)\n        # Plain pipe determinant: | a b | | c d | | e f | → PSV (each row on its own line)\n        text = _PIPE_MATRIX.sub(lambda m: \'\\n\'.join(re.findall(r\'\\|\\s*([^|\\n]+?)\\s*\\|\', m.group(0))), text)\n\n    # 3. Cyrillic/Latin lookalikes → Cyrillic\n    text = \'\'.join(_LATIN_TO_CYRILLIC.get(ch, ch) for ch in text)\n\n    # 4. Dashes → hyphen-minus\n    text = _DASHES.sub(\'-\', text)\n\n    # 5. Filler dashes/underscores → 3 chars\n    text = _FILLERS.sub(\'___\', text)\n\n    # 6. Quotes\n    text = _QUOTES_DOUBLE.sub(\'"\', text)\n    text = _QUOTES_SINGLE.sub("\'", text)\n\n    # 7. Whitespace: NBSP and exotic spaces → regular space, collapse multiples\n    text = _MULTI_SPACE.sub(\' \', text)\n\n    # 8. Spaces inside parentheses\n    text = _SPACE_IN_PARENS.sub(\'(\', text)\n    text = _SPACE_IN_PARENS_R.sub(\')\', text)\n    # 8b. Strip trailing space introduced by `\\sin ` / `\\lim ` etc. when\n    # followed by "(" or sub/superscript marker.\n    text = _SPACE_BEFORE_PAREN.sub(\'(\', text)\n    text = _SPACE_BEFORE_SUBSUPER.sub(r\'\\1\', text)\n\n    # 9. Table-specific: PSV cleanup (LaTeX table already converted in step 2)\n    if region_type == "table":\n        text = _PSV_PIPE.sub(\'|\', text)\n        lines = text.split(\'\\n\')\n        lines = [line.strip(\'|\').strip() for line in lines]\n        text = \'\\n\'.join(line for line in lines if line)\n\n    # 10. Strip leading/trailing whitespace\n    text = text.strip()\n\n    return text\n\n\n# ── Levenshtein distance (pure Python, no external deps) ──────\n\ndef _levenshtein(s1: str, s2: str) -> int:\n    if len(s1) < len(s2):\n        return _levenshtein(s2, s1)\n    if len(s2) == 0:\n        return len(s1)\n    prev = list(range(len(s2) + 1))\n    for i, c1 in enumerate(s1):\n        curr = [i + 1]\n        for j, c2 in enumerate(s2):\n            curr.append(min(\n                prev[j + 1] + 1,\n                curr[j] + 1,\n                prev[j] + (c1 != c2),\n            ))\n        prev = curr\n    return prev[-1]\n\n\n# ── IoU ───────────────────────────────────────────────────────\n\ndef _compute_iou(bbox1, bbox2):\n    x1 = max(bbox1[0], bbox2[0])\n    y1 = max(bbox1[1], bbox2[1])\n    x2 = min(bbox1[2], bbox2[2])\n    y2 = min(bbox1[3], bbox2[3])\n    if x2 <= x1 or y2 <= y1:\n        return 0.0\n    intersection = (x2 - x1) * (y2 - y1)\n    area1 = max(0, bbox1[2] - bbox1[0]) * max(0, bbox1[3] - bbox1[1])\n    area2 = max(0, bbox2[2] - bbox2[0]) * max(0, bbox2[3] - bbox2[1])\n    union = area1 + area2 - intersection\n    return intersection / union if union > 0 else 0.0\n\n\n# ── Greedy IoU matching ───────────────────────────────────────\n\ndef _greedy_match(gt_regions, pred_regions, threshold=0.5):\n    pairs = []\n    for gi, g in enumerate(gt_regions):\n        for pi, p in enumerate(pred_regions):\n            iou = _compute_iou(g["bbox"], p["bbox"])\n            if iou >= threshold:\n                pairs.append((iou, gi, pi))\n    pairs.sort(key=lambda x: -x[0])\n\n    matched = []\n    used_gt, used_pred = set(), set()\n    for iou, gi, pi in pairs:\n        if gi not in used_gt and pi not in used_pred:\n            matched.append((gi, pi))\n            used_gt.add(gi)\n            used_pred.add(pi)\n\n    unmatched_gt = [i for i in range(len(gt_regions)) if i not in used_gt]\n    unmatched_pred = [i for i in range(len(pred_regions)) if i not in used_pred]\n    return matched, unmatched_gt, unmatched_pred\n\n\n# ── Scorable check ────────────────────────────────────────────\n\ndef _is_scorable(region):\n    if region.get("type", "handwritten") in ("image", "graph"):\n        return False\n    if region.get("language", "uk") == "other":\n        return False\n    if region.get("legibility", "legible") == "illegible":\n        return False\n    return True\n\n\n# ── Page text builder ─────────────────────────────────────────\n\ndef _build_page_text(regions, normalize=False, drop_indices=None):\n    """Build concatenated page text from regions.\n\n    `drop_indices` (optional): set of region indices to additionally exclude\n    beyond the standard _is_scorable filter. Used on the prediction side to\n    drop pred regions that match a non-scorable GT region (otherwise their\n    text inflates pred_page asymmetrically vs gt_page).\n    """\n    drop = drop_indices or set()\n    scorable = [\n        r for i, r in enumerate(regions)\n        if _is_scorable(r) and i not in drop\n    ]\n    # Bucketed reading order: cluster regions whose center_y is within ~15px\n    # (half a typical handwritten line height), then left-to-right by center_x.\n    # Stabilizes ordering when detection splits one GT line into multiple bboxes\n    # with slightly different top_y values, which would otherwise scramble the\n    # page-level text concatenation.\n    scorable.sort(key=lambda r: (\n        ((r["bbox"][1] + r["bbox"][3]) / 2) // 15,\n        (r["bbox"][0] + r["bbox"][2]) / 2,\n    ))\n    if normalize:\n        return "\\n".join(\n            _normalize_text(r.get("text", ""), r.get("type", "handwritten"))\n            for r in scorable\n        )\n    return "\\n".join(r.get("text", "") for r in scorable)\n\n\n# ── Parse regions JSON ────────────────────────────────────────\n\nVALID_TYPES = {"handwritten", "printed", "formula", "table", "annotation", "image", "graph"}\n\n\ndef _parse_regions(regions_str, image_name, is_submission=True):\n    label = "Submission" if is_submission else "Solution"\n    if pd.isna(regions_str) or regions_str == "":\n        return []\n    try:\n        regions = json.loads(regions_str)\n    except (json.JSONDecodeError, TypeError) as e:\n        raise ParticipantVisibleError(\n            f\'{label} for image "{image_name}": invalid JSON in regions column. Error: {e}\'\n        )\n    if not isinstance(regions, list):\n        raise ParticipantVisibleError(\n            f\'{label} for image "{image_name}": regions must be a JSON list, got {type(regions).__name__}\'\n        )\n    for i, r in enumerate(regions):\n        if not isinstance(r, dict):\n            raise ParticipantVisibleError(\n                f\'{label} for image "{image_name}", region {i}: must be a JSON object\'\n            )\n        if "bbox" not in r:\n            raise ParticipantVisibleError(\n                f\'{label} for image "{image_name}", region {i}: missing "bbox" field\'\n            )\n        bbox = r["bbox"]\n        if not isinstance(bbox, list) or len(bbox) != 4:\n            raise ParticipantVisibleError(\n                f\'{label} for image "{image_name}", region {i}: bbox must be [x1, y1, x2, y2]\'\n            )\n        rtype = r.get("type", "handwritten")\n        if is_submission and rtype not in VALID_TYPES:\n            raise ParticipantVisibleError(\n                f\'{label} for image "{image_name}", region {i}: invalid type "{rtype}". \'\n                f\'Must be one of: {", ".join(sorted(VALID_TYPES))}\'\n            )\n        # Defaults\n        r.setdefault("type", "handwritten")\n        r.setdefault("language", "uk")\n        r.setdefault("legibility", "legible")\n        r.setdefault("text", "")\n    return regions\n\n\n# ── Main scoring function ────────────────────────────────────\n\ndef score(\n    solution: pd.DataFrame,\n    submission: pd.DataFrame,\n    row_id_column_name: str,\n    w_det: float = 0.15,\n    w_cls: float = 0.05,\n    w_cer: float = 0.30,\n    w_page: float = 0.50,\n) -> float:\n    """\n    Ukrainian Handwritten Text Recognition (HTR) Competition Metric.\n\n    Evaluates end-to-end document understanding: region detection,\n    classification, and text transcription.\n\n    Score = 0.15 * Detection_F1 + 0.05 * ClassAcc + 0.30 * (1-CER) + 0.50 * (1-PageCER)\n\n    Components:\n      - Detection F1 (0.15): type-agnostic bbox matching at IoU >= 0.5\n      - Classification Accuracy (0.05): correct region type among IoU-matched pairs\n      - CER (0.30): per-region Character Error Rate on matched scorable regions\n      - Page CER (0.50): full-page text comparison, agnostic to bbox granularity\n\n    Score range: 0.0 to 1.0 (higher is better).\n\n    Submission: CSV with columns `image` and `regions`.\n    `regions` is a JSON list of detected regions per image:\n        [{"bbox": [x1,y1,x2,y2], "type": "handwritten", "text": "..."}]\n\n    Region types: handwritten, printed, formula, table, annotation, image, graph.\n    Use [] for images with no detections. All test images must be present.\n\n    Text normalization (applied to both GT and predictions before CER):\n      - Cyrillic/Latin lookalikes unified (Latin c → Cyrillic с)\n      - Dash types unified (em/en-dash → hyphen)\n      - Whitespace collapsed, quotes normalized\n      - Strikethrough markers removed: ~~old~~{new} → new\n      - Formula: x_{3} → x_3, x² → x^2\n      - Table: whitespace around pipes stripped\n\n    Regions excluded from CER (GT attributes, not required from participants):\n      type=image/graph, language=other, legibility=illegible\n\n    >>> import pandas as pd\n    >>> sol = pd.DataFrame({"image": ["a.jpg"], "regions": [\'[{"bbox":[0,0,100,50],"type":"handwritten","text":"hello"}]\']})\n    >>> sub = pd.DataFrame({"image": ["a.jpg"], "regions": [\'[{"bbox":[0,0,100,50],"type":"handwritten","text":"hello"}]\']})\n    >>> score(sol, sub, "image")\n    1.0\n    """\n    # Validate columns\n    if "regions" not in submission.columns:\n        raise ParticipantVisibleError(\n            \'Submission must have a "regions" column containing JSON-encoded region predictions.\'\n        )\n    if "regions" not in solution.columns:\n        raise ParticipantVisibleError(\'Solution is missing "regions" column.\')\n\n    # Check all solution images are in submission\n    sol_images = set(solution[row_id_column_name])\n    sub_images = set(submission[row_id_column_name])\n    missing = sol_images - sub_images\n    if missing:\n        examples = sorted(missing)[:5]\n        raise ParticipantVisibleError(\n            f\'Submission is missing {len(missing)} image(s). Examples: {examples}. \'\n            f\'Include all test images, even those with no predictions (use empty regions: []).\'\n        )\n\n    # Build lookup\n    sub_lookup = {}\n    for _, row in submission.iterrows():\n        img = row[row_id_column_name]\n        sub_lookup[img] = _parse_regions(row["regions"], img, is_submission=True)\n\n    # Accumulators\n    all_det_tp = 0\n    all_det_fp = 0\n    all_det_fn = 0\n    all_class_correct = 0\n    all_class_total = 0\n    all_cer_values = []\n    all_page_cers = []\n\n    for _, row in solution.iterrows():\n        img = row[row_id_column_name]\n        gt = _parse_regions(row["regions"], img, is_submission=False)\n        pred = sub_lookup.get(img, [])\n\n        # Match by IoU\n        matched, unmatched_gt, unmatched_pred = _greedy_match(gt, pred, threshold=0.5)\n\n        # Detection F1 (type-agnostic: measures bbox quality only)\n        all_det_tp += len(matched)\n        all_det_fp += len(unmatched_pred)\n        all_det_fn += len(unmatched_gt)\n\n        # Classification Accuracy\n        for gi, pi in matched:\n            all_class_total += 1\n            if gt[gi]["type"] == pred[pi]["type"]:\n                all_class_correct += 1\n\n        # CER (per-region, with text normalization)\n        for gi, pi in matched:\n            if _is_scorable(gt[gi]):\n                rtype = gt[gi].get("type", "handwritten")\n                gt_text = _normalize_text(gt[gi].get("text", ""), rtype)\n                pred_text = _normalize_text(pred[pi].get("text", ""), rtype)\n                cer_i = _levenshtein(pred_text, gt_text) / max(len(gt_text), 1)\n                all_cer_values.append(cer_i)\n\n        # Page CER (with text normalization).\n        # Drop pred regions matched to a non-scorable GT region — otherwise\n        # their text would inflate pred_page while GT side excludes them.\n        pred_drop = {pi for gi, pi in matched if not _is_scorable(gt[gi])}\n        gt_page = _build_page_text(gt, normalize=True)\n        pred_page = _build_page_text(pred, normalize=True, drop_indices=pred_drop)\n        if len(gt_page) > 0:\n            page_cer_i = _levenshtein(pred_page, gt_page) / len(gt_page)\n            all_page_cers.append(page_cer_i)\n\n    # Aggregate\n    det_prec = all_det_tp / max(all_det_tp + all_det_fp, 1)\n    det_rec = all_det_tp / max(all_det_tp + all_det_fn, 1)\n    det_f1 = 2 * det_prec * det_rec / max(det_prec + det_rec, 1e-9)\n\n    class_acc = all_class_correct / max(all_class_total, 1)\n\n    # Default CER = 1.0 (worst) when no regions matched — prevents free score for empty submissions\n    cer = sum(all_cer_values) / len(all_cer_values) if all_cer_values else 1.0\n    page_cer = sum(all_page_cers) / len(all_page_cers) if all_page_cers else 1.0\n\n    final_score = (\n        w_det * det_f1\n        + w_cls * class_acc\n        + w_cer * max(0.0, 1.0 - cer)\n        + w_page * max(0.0, 1.0 - page_cer)\n    )\n\n    return float(final_score)\n\ndef score_detailed(\n    solution: pd.DataFrame,\n    submission: pd.DataFrame,\n    row_id_column_name: str,\n    w_det: float = 0.15,\n    w_cls: float = 0.05,\n    w_cer: float = 0.30,\n    w_page: float = 0.50,\n) -> dict:\n    """\n    Same metric as `score()` but returns a dict with all component scores.\n\n    Use this locally to debug your submission — you will see which component\n    (detection, classification, per-region CER, or page CER) is dragging\n    the score down.\n\n    Not used by Kaggle — Kaggle calls `score()` which returns a single float.\n    """\n    if "regions" not in submission.columns:\n        raise ParticipantVisibleError(\n            \'Submission must have a "regions" column containing JSON-encoded region predictions.\'\n        )\n    if "regions" not in solution.columns:\n        raise ParticipantVisibleError(\'Solution is missing "regions" column.\')\n\n    sol_images = set(solution[row_id_column_name])\n    sub_images = set(submission[row_id_column_name])\n    missing = sol_images - sub_images\n    if missing:\n        examples = sorted(missing)[:5]\n        raise ParticipantVisibleError(\n            f\'Submission is missing {len(missing)} image(s). Examples: {examples}.\'\n        )\n\n    sub_lookup = {\n        row[row_id_column_name]: _parse_regions(row["regions"], row[row_id_column_name], is_submission=True)\n        for _, row in submission.iterrows()\n    }\n\n    all_det_tp = all_det_fp = all_det_fn = 0\n    all_class_correct = all_class_total = 0\n    all_cer_values, all_page_cers = [], []\n\n    for _, row in solution.iterrows():\n        img = row[row_id_column_name]\n        gt = _parse_regions(row["regions"], img, is_submission=False)\n        pred = sub_lookup.get(img, [])\n        matched, unmatched_gt, unmatched_pred = _greedy_match(gt, pred, threshold=0.5)\n\n        all_det_tp += len(matched)\n        all_det_fp += len(unmatched_pred)\n        all_det_fn += len(unmatched_gt)\n\n        for gi, pi in matched:\n            all_class_total += 1\n            if gt[gi]["type"] == pred[pi]["type"]:\n                all_class_correct += 1\n\n        for gi, pi in matched:\n            if _is_scorable(gt[gi]):\n                rtype = gt[gi].get("type", "handwritten")\n                gt_text = _normalize_text(gt[gi].get("text", ""), rtype)\n                pred_text = _normalize_text(pred[pi].get("text", ""), rtype)\n                cer_i = _levenshtein(pred_text, gt_text) / max(len(gt_text), 1)\n                all_cer_values.append(cer_i)\n\n        # Drop pred regions matched to non-scorable GT (avoid asymmetric pred_page inflation)\n        pred_drop = {pi for gi, pi in matched if not _is_scorable(gt[gi])}\n        gt_page = _build_page_text(gt, normalize=True)\n        pred_page = _build_page_text(pred, normalize=True, drop_indices=pred_drop)\n        if len(gt_page) > 0:\n            all_page_cers.append(_levenshtein(pred_page, gt_page) / len(gt_page))\n\n    det_prec = all_det_tp / max(all_det_tp + all_det_fp, 1)\n    det_rec = all_det_tp / max(all_det_tp + all_det_fn, 1)\n    det_f1 = 2 * det_prec * det_rec / max(det_prec + det_rec, 1e-9)\n    class_acc = all_class_correct / max(all_class_total, 1)\n    cer = sum(all_cer_values) / len(all_cer_values) if all_cer_values else 1.0\n    page_cer = sum(all_page_cers) / len(all_page_cers) if all_page_cers else 1.0\n\n    composite = (\n        w_det * det_f1\n        + w_cls * class_acc\n        + w_cer * max(0.0, 1.0 - cer)\n        + w_page * max(0.0, 1.0 - page_cer)\n    )\n\n    return {\n        "composite_score": float(composite),\n        "detection_f1": float(det_f1),\n        "detection_precision": float(det_prec),\n        "detection_recall": float(det_rec),\n        "classification_accuracy": float(class_acc),\n        "region_cer": float(cer),\n        "page_cer": float(page_cer),\n        "n_images": int(len(solution)),\n        "n_matched_regions": int(all_det_tp),\n        "n_false_positives": int(all_det_fp),\n        "n_false_negatives": int(all_det_fn),\n    }\n\n\n# ── CLI for local debugging ───────────────────────────────────\n# NOTE: kept as a function (not a top-level __main__ block) so that\n# importing this file in a notebook / Kaggle metric uploader does not\n# trigger argparse on a kernel-launcher cmdline.\n\ndef _cli_main():\n    import argparse\n    parser = argparse.ArgumentParser(\n        description="RUKOPYS scoring — run locally to see component breakdown",\n    )\n    parser.add_argument("--solution", required=True, help="Path to ground-truth CSV (image, regions)")\n    parser.add_argument("--submission", required=True, help="Path to your submission CSV (image, regions)")\n    parser.add_argument("--row-id", default="image", help="Row ID column name (default: image)")\n    args = parser.parse_args()\n\n    sol = pd.read_csv(args.solution)\n    sub = pd.read_csv(args.submission)\n\n    r = score_detailed(sol, sub, args.row_id)\n\n    print()\n    print(f"  Images evaluated       : {r[\'n_images\']}")\n    print(f"  Matched regions (IoU≥.5): {r[\'n_matched_regions\']}")\n    print(f"  False positives        : {r[\'n_false_positives\']}")\n    print(f"  False negatives        : {r[\'n_false_negatives\']}")\n    print()\n    print(f"  Detection F1           : {r[\'detection_f1\']:.4f}   (precision {r[\'detection_precision\']:.3f} / recall {r[\'detection_recall\']:.3f})")\n    print(f"  Classification accuracy: {r[\'classification_accuracy\']:.4f}")\n    print(f"  Region CER             : {r[\'region_cer\']:.4f}   → score {1-r[\'region_cer\']:.4f}")\n    print(f"  Page CER               : {r[\'page_cer\']:.4f}   → score {1-r[\'page_cer\']:.4f}")\n    print(f"  ──────────────────────────────────────────────────")\n    print(f"  Composite score        : {r[\'composite_score\']:.4f}")\n    print()\n\n\nif __name__ == "__main__":\n    import sys\n    # Only auto-run CLI if --solution arg present; skips Jupyter/Colab\n    # where __name__ == "__main__" but sys.argv is a kernel launcher.\n    if any(a == "--solution" for a in sys.argv[1:]):\n        _cli_main()\n'
metric_path = EVALUATOR_ROOT / "kaggle_metric.py"
metric_path.parent.mkdir(parents=True, exist_ok=True)
metric_path.write_text(KAGGLE_METRIC_SOURCE, encoding="utf-8")
if str(metric_path.parent) not in sys.path:
    sys.path.insert(0, str(metric_path.parent))
print(f"Wrote {metric_path} from official BTC notebook source")


In [ ]:
import csv
import json
import math
import re
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

from kaggle_metric import (
    ParticipantVisibleError,
    _greedy_match,
    _is_scorable,
    _levenshtein,
    _normalize_text,
    _parse_regions,
    score_detailed,
)


def read_jsonl(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def clean_region_for_solution(region):
    cleaned = {
        "bbox": region.get("bbox", [0, 0, 0, 0]),
        "type": str(region.get("type") or "handwritten"),
        "text": str(region.get("text") or ""),
    }
    for key in ["language", "legibility"]:
        if key in region:
            cleaned[key] = region[key]
    return cleaned


def build_solution_df(gt_jsonl):
    records = read_jsonl(gt_jsonl)
    rows = []
    for record in records:
        image = Path(str(record.get("file_name") or record.get("image") or "")).name
        regions = [clean_region_for_solution(r) for r in record.get("regions", [])]
        rows.append({"image": image, "regions": json.dumps(regions, ensure_ascii=False)})
    return pd.DataFrame(rows), records


def infer_run_id(path):
    path = Path(path)
    stem = path.stem
    if stem.endswith("_submission"):
        return stem[: -len("_submission")]
    if path.name == "submission.csv":
        return path.parent.name
    return stem


def label_for_run(run_id):
    if run_id in RUN_LABEL_OVERRIDES:
        return RUN_LABEL_OVERRIDES[run_id]
    label = run_id
    label = re.sub(r"^\d+[_-]", "", label)
    label = label.replace("_", " ").replace("-", " ")
    return label.strip()


def discover_submission_csvs(search_dirs, extra_files):
    paths = []
    for item in search_dirs:
        root = Path(item)
        if not root.exists():
            continue
        if root.is_file() and root.suffix.lower() == ".csv":
            paths.append(root)
            continue
        for p in root.rglob("*.csv"):
            if p.suffix.lower() != ".csv":
                continue
            paths.append(p)
    for item in extra_files:
        p = Path(item)
        if p.exists():
            paths.append(p)
    unique = []
    seen = set()
    for p in paths:
        rp = p.resolve()
        if rp not in seen:
            seen.add(rp)
            unique.append(p)
    return sorted(unique, key=lambda p: str(p))


def safe_read_submission(path):
    df = pd.read_csv(path)
    if "image" not in df.columns or "regions" not in df.columns:
        raise ValueError(f"{path} phải có cột image và regions")
    df = df[["image", "regions"]].copy()
    df["image"] = df["image"].astype(str)
    df["regions"] = df["regions"].fillna("[]").astype(str)
    return df


def sequence_levenshtein(seq1, seq2):
    if len(seq1) < len(seq2):
        return sequence_levenshtein(seq2, seq1)
    if len(seq2) == 0:
        return len(seq1)
    prev = list(range(len(seq2) + 1))
    for i, c1 in enumerate(seq1):
        curr = [i + 1]
        for j, c2 in enumerate(seq2):
            curr.append(min(prev[j + 1] + 1, curr[j] + 1, prev[j] + (c1 != c2)))
        prev = curr
    return prev[-1]


def word_error_rate(pred_text, gt_text):
    pred_words = str(pred_text or "").split()
    gt_words = str(gt_text or "").split()
    return sequence_levenshtein(pred_words, gt_words) / max(len(gt_words), 1)


def fmt_float(x):
    if pd.isna(x):
        return ""
    return f"{float(x):.{ROUND_DIGITS}f}"


def markdown_cell(value):
    if pd.isna(value):
        text = ""
    else:
        text = str(value)
    return text.replace("|", "\\|").replace("\n", "<br>")


def save_markdown_table(df, path):
    path = Path(path)
    if df.empty:
        path.write_text("_No rows._\n", encoding="utf-8")
        return
    columns = list(df.columns)
    header = "| " + " | ".join(markdown_cell(c) for c in columns) + " |"
    sep = "| " + " | ".join("---" for _ in columns) + " |"
    body = []
    for _, row in df.iterrows():
        body.append("| " + " | ".join(markdown_cell(row[c]) for c in columns) + " |")
    path.write_text("\n".join([header, sep, *body]) + "\n", encoding="utf-8")


def ordered_by_label(df, label_col, order):
    if df.empty:
        return df
    rank = {label: i for i, label in enumerate(order)}
    return df.assign(_rank=df[label_col].map(lambda x: rank.get(x, 999))).sort_values(["_rank", label_col]).drop(columns=["_rank"])


def hpa_region_details(solution_df, submission_df, run_id, label, gt_records):
    sub_lookup = {
        row[ROW_ID_COLUMN]: _parse_regions(row["regions"], row[ROW_ID_COLUMN], is_submission=True)
        for _, row in submission_df.iterrows()
    }
    source_by_image = {}
    for record in gt_records:
        image = Path(str(record.get("file_name") or record.get("image") or "")).name
        source_by_image[image] = str(record.get("source") or "unknown")

    rows = []
    for _, row in solution_df.iterrows():
        image = row[ROW_ID_COLUMN]
        gt = _parse_regions(row["regions"], image, is_submission=False)
        pred = sub_lookup.get(image, [])
        matched, _, _ = _greedy_match(gt, pred, threshold=IOU_THRESHOLD)
        for gi, pi in matched:
            gt_region = gt[gi]
            pred_region = pred[pi]
            rtype = gt_region.get("type", "handwritten")
            if rtype not in HPA_TYPES:
                continue
            if not _is_scorable(gt_region):
                continue
            gt_text_raw = gt_region.get("text", "")
            pred_text_raw = pred_region.get("text", "")
            gt_text = _normalize_text(gt_text_raw, rtype)
            pred_text = _normalize_text(pred_text_raw, rtype)
            cer = _levenshtein(pred_text, gt_text) / max(len(gt_text), 1)
            wer = word_error_rate(pred_text, gt_text)
            rows.append({
                "run_id": run_id,
                "label": label,
                "image": image,
                "source": source_by_image.get(image, "unknown"),
                "type": rtype,
                "cer": float(cer),
                "wer": float(wer),
                "exact_match": bool(pred_text == gt_text),
                "gt_text": gt_text_raw,
                "pred_text": pred_text_raw,
                "gt_norm": gt_text,
                "pred_norm": pred_text,
            })
    return pd.DataFrame(rows)


def summarize_hpa_details(details_df):
    if details_df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    summary_rows = []
    per_type_rows = []
    per_source_rows = []
    for run_id, part in details_df.groupby("run_id", sort=False):
        label = part["label"].iloc[0]
        summary_rows.append({
            "run_id": run_id,
            "label": label,
            "N": int(len(part)),
            "overall_cer": float(part["cer"].mean()),
            "overall_wer": float(part["wer"].mean()),
            "exact_match": float(part["exact_match"].mean()),
            "handwritten_cer": float(part.loc[part["type"].eq("handwritten"), "cer"].mean()),
            "printed_cer": float(part.loc[part["type"].eq("printed"), "cer"].mean()),
            "annotation_cer": float(part.loc[part["type"].eq("annotation"), "cer"].mean()),
        })
        for rtype in HPA_TYPES:
            sub = part[part["type"].eq(rtype)]
            per_type_rows.append({
                "run_id": run_id,
                "label": label,
                "type": rtype,
                "N": int(len(sub)),
                "cer": float(sub["cer"].mean()) if not sub.empty else math.nan,
                "wer": float(sub["wer"].mean()) if not sub.empty else math.nan,
                "exact_match": float(sub["exact_match"].mean()) if not sub.empty else math.nan,
            })
        for source, sub in part.groupby("source"):
            per_source_rows.append({
                "run_id": run_id,
                "label": label,
                "source": source,
                "N": int(len(sub)),
                "cer": float(sub["cer"].mean()),
                "wer": float(sub["wer"].mean()),
                "exact_match": float(sub["exact_match"].mean()),
            })
    return pd.DataFrame(summary_rows), pd.DataFrame(per_type_rows), pd.DataFrame(per_source_rows)


def build_paper_module_table(hpa_summary_df, hpa_per_type_df):
    rows = []
    if hpa_summary_df.empty:
        return pd.DataFrame(columns=["Component", "Split / Group", "Metric", "N", "Value"])
    final = hpa_summary_df[hpa_summary_df["run_id"].eq(FINAL_HPA_RUN_ID)]
    if final.empty:
        final = hpa_summary_df.head(1)
    final_row = final.iloc[0]
    run_id = final_row["run_id"]
    rows.extend([
        {"Component": "TrOCR HPA", "Split / Group": "overall", "Metric": "CER", "N": int(final_row["N"]), "Value": fmt_float(final_row["overall_cer"])},
        {"Component": "TrOCR HPA", "Split / Group": "overall", "Metric": "WER", "N": int(final_row["N"]), "Value": fmt_float(final_row["overall_wer"])},
        {"Component": "TrOCR HPA", "Split / Group": "overall", "Metric": "Exact Match", "N": int(final_row["N"]), "Value": fmt_float(final_row["exact_match"])},
    ])
    per_type = hpa_per_type_df[hpa_per_type_df["run_id"].eq(run_id)]
    for rtype in HPA_TYPES:
        sub = per_type[per_type["type"].eq(rtype)]
        if sub.empty:
            rows.append({"Component": "TrOCR HPA", "Split / Group": rtype, "Metric": "CER", "N": 0, "Value": ""})
        else:
            row = sub.iloc[0]
            rows.append({"Component": "TrOCR HPA", "Split / Group": rtype, "Metric": "CER", "N": int(row["N"]), "Value": fmt_float(row["cer"])})
    return pd.DataFrame(rows)


def build_hpa_ablation_table(hpa_summary_df):
    if hpa_summary_df.empty:
        return pd.DataFrame()
    table = hpa_summary_df[["label", "N", "overall_cer", "handwritten_cer", "printed_cer", "annotation_cer", "overall_wer", "exact_match"]].copy()
    table = ordered_by_label(table, "label", HPA_ABLATION_ORDER)
    for c in ["overall_cer", "handwritten_cer", "printed_cer", "annotation_cer", "overall_wer", "exact_match"]:
        table[c] = table[c].map(fmt_float)
    return table


def build_hpa_per_type_paper_table(hpa_per_type_df):
    columns = ["Type", "N", "CER", "WER", "Exact Match"]
    if hpa_per_type_df.empty:
        return pd.DataFrame(columns=columns)
    selected = hpa_per_type_df[hpa_per_type_df["run_id"] == FINAL_HPA_RUN_ID].copy()
    if selected.empty:
        first_run = hpa_per_type_df["run_id"].iloc[0]
        selected = hpa_per_type_df[hpa_per_type_df["run_id"] == first_run].copy()
    rows = []
    for rtype in HPA_TYPES:
        row_df = selected[selected["type"] == rtype]
        if row_df.empty:
            rows.append({"Type": rtype, "N": 0, "CER": "", "WER": "", "Exact Match": ""})
            continue
        row = row_df.iloc[0]
        rows.append({
            "Type": rtype,
            "N": int(row["N"]),
            "CER": fmt_float(row["cer"]),
            "WER": fmt_float(row["wer"]),
            "Exact Match": fmt_float(row["exact_match"]),
        })
    return pd.DataFrame(rows, columns=columns)


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
solution_df, GT_RECORDS = build_solution_df(GT_JSONL)
solution_csv = OUTPUT_DIR / "solution_from_test_jsonl.csv"
solution_df.to_csv(solution_csv, index=False)
print(f"Ghi solution CSV: {solution_csv} rows={len(solution_df)}")

submission_paths = discover_submission_csvs(SUBMISSION_SEARCH_DIRS, EXTRA_SUBMISSION_FILES)
print(f"Tìm thấy {len(submission_paths)} HPA/TrOCR submission CSV")
for p in submission_paths:
    print("-", p)

all_official_rows = []
all_details = []
errors = []

for path in submission_paths:
    run_id = infer_run_id(path)
    label = label_for_run(run_id)
    print(f"\nĐang chấm HPA: {run_id} -> {path}")
    try:
        sub_df = safe_read_submission(path)
        detail = score_detailed(solution_df, sub_df, ROW_ID_COLUMN)
        hpa_details = hpa_region_details(solution_df, sub_df, run_id, label, GT_RECORDS)
        all_details.append(hpa_details)
        all_official_rows.append({
            "run_id": run_id,
            "label": label,
            "path": str(path),
            "official_score": detail["composite_score"],
            "det_f1": detail["detection_f1"],
            "class_acc": detail["classification_accuracy"],
            "region_cer": detail["region_cer"],
            "page_cer": detail["page_cer"],
            "matched_regions": detail["n_matched_regions"],
            "false_positives": detail["n_false_positives"],
            "false_negatives": detail["n_false_negatives"],
            "hpa_N": int(len(hpa_details)),
            "hpa_cer": float(hpa_details["cer"].mean()) if not hpa_details.empty else math.nan,
            "hpa_wer": float(hpa_details["wer"].mean()) if not hpa_details.empty else math.nan,
            "hpa_exact_match": float(hpa_details["exact_match"].mean()) if not hpa_details.empty else math.nan,
        })
        print(f"  HPA N={len(hpa_details)} CER={hpa_details['cer'].mean():.4f} WER={hpa_details['wer'].mean():.4f} Exact={hpa_details['exact_match'].mean():.4f}")
    except Exception as exc:
        print(f"  LỖI: {exc}")
        errors.append({"run_id": run_id, "path": str(path), "error": repr(exc)})

official_df = pd.DataFrame(all_official_rows)
details_df = pd.concat(all_details, ignore_index=True) if all_details else pd.DataFrame()
hpa_summary_df, hpa_per_type_df, hpa_per_source_df = summarize_hpa_details(details_df)
errors_df = pd.DataFrame(errors)

paths = {
    "all_hpa_submission_metrics.csv": official_df,
    "hpa_region_details.csv": details_df,
    "hpa_metric_summary.csv": hpa_summary_df,
    "hpa_per_type_metrics.csv": hpa_per_type_df,
    "hpa_per_source_metrics.csv": hpa_per_source_df,
    "evaluation_errors.csv": errors_df,
}
for filename, df in paths.items():
    out_path = OUTPUT_DIR / filename
    df.to_csv(out_path, index=False)
    print("Ghi", out_path)


In [ ]:
paper_module_table = build_paper_module_table(hpa_summary_df, hpa_per_type_df)
paper_hpa_ablation_table = build_hpa_ablation_table(hpa_summary_df)
paper_hpa_per_type_table = build_hpa_per_type_paper_table(hpa_per_type_df)

paper_outputs = {
    "paper_hpa_module_results.csv": paper_module_table,
    "paper_hpa_ablation.csv": paper_hpa_ablation_table,
    "paper_hpa_per_type_results.csv": paper_hpa_per_type_table,
}
for filename, df in paper_outputs.items():
    csv_path = OUTPUT_DIR / filename
    md_path = OUTPUT_DIR / filename.replace(".csv", ".md")
    df.to_csv(csv_path, index=False)
    save_markdown_table(df, md_path)
    print("Ghi", csv_path)
    print("Ghi", md_path)

print("\n=== HPA module rows for tab:module-results ===")
display(paper_module_table)

print("\n=== HPA ablation table ===")
display(paper_hpa_ablation_table)

print("\n=== HPA per-type table ===")
display(paper_hpa_per_type_table)

if not details_df.empty:
    worst = details_df.sort_values("cer", ascending=False).head(50)
    worst_path = OUTPUT_DIR / "hpa_worst_cer_examples_top50.csv"
    worst.to_csv(worst_path, index=False)
    print("\nGhi ví dụ lỗi CER cao:", worst_path)
    display(worst[["run_id", "image", "type", "cer", "wer", "gt_text", "pred_text"]].head(10))
